##Install and imports

### install

In [1]:
# !pip install bitsandbytes
# !pip install transformers==4.40.2
# !pip install peft==0.11.1
# !pip install accelerate==0.30.1

### import

In [2]:
import os
import json
import tqdm
import sys

import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

True
NVIDIA GeForce RTX 5060 Ti
17.1 GB


## Load model

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "HuggingFaceH4/zephyr-7b-beta"
tokenizer = AutoTokenizer.from_pretrained(model_name, truncation=True, padding=True, padding_side="left", maximum_length=2048, model_max_length=2048)
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_4bit=True, device_map='auto')
tokenizer.pad_token = tokenizer.eos_token
model.generation_config.pad_token_id = model.generation_config.eos_token_id

c:\Users\Matiss\anaconda3\envs\py312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
W0430 13:11:48.016000 23012 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

## Query expansion

In [4]:
# first up load the data that is in /Data folder

# define paths
DATA_DIR = "Data"

QUERIES_PATH = os.path.join(DATA_DIR, "queries.tsv")
CANDIDATES_PATH = os.path.join(DATA_DIR, "candidates.tsv")
QRELS_PATH = os.path.join(DATA_DIR, "qrels.txt")

# load queruies
queries = {}
with open(QUERIES_PATH, "r") as f:
    for line in f:
        query_id, query = line.strip().split("\t")
        queries[query_id] = query
        
        
# load candidates
candidates = {}
with open(CANDIDATES_PATH, "r") as f:
    for line in f:
        parts = line.strip().split("\t")
        query_id, docid, doctext = parts[0], parts[1], parts[2]
        if query_id not in candidates:
            candidates[query_id] = []
        candidates[query_id].append((docid, doctext))
        

# load qrels
qrels = {}
with open(QRELS_PATH, "r") as f:
    for line in f:
        parts = line.strip().split() # not a tab-separated file (tsv) so regular white space split
        query_id, docid, relevance = parts[0], parts[2], int(parts[3]), # skip 2nd column and relevance in integers
        if query_id not in qrels:
            qrels[query_id] = {}
        qrels[query_id][docid] = relevance
        
# print(f"#queries    {len(queries)}") #200
# print(f"#query ids with candidates {len(candidates)}") # 200
# print(f"#query ids with qrels      {len(qrels)}") # 43

# we will only work with the 43 queries that have qrels, so filter candidates and queries to those
qrel_query_ids = set(qrels.keys())
queries = {query_id: query_text for query_id, query_text in queries.items() if query_id in qrel_query_ids}
candidates = {query_id: docs for query_id, docs in candidates.items() if query_id in qrel_query_ids}


 # check
assert len(queries) == 43
assert len(candidates) == 43
assert len(qrels) == 43
sample_query_id = list(queries.keys())[0]
print(f"\nsample query [{sample_query_id}]: {queries[sample_query_id]}")
print(f"{len(candidates[sample_query_id])} candidate docs")
print(f"{len(qrels.get(sample_query_id, {}))} qrel judgements")


sample query [156493]: do goldfish grow
1000 candidate docs
300 qrel judgements


In [5]:
# query expansion pipeline
# we build the CoT pipeline for each query and run it through the LLM

def build_cot_prompt(query):
    """Build the CoT prompt from Table 3 of the paper https://arxiv.org/pdf/2305.03653

    Args:
        query (str): Query to fit in the prompt

    Returns:
        output: Full prompt with the query
    """
    return f"Answer the following query:\n{query}\nGive the rationale before answering" 


def generate_expansions(queries, model, tokenizer, max_new_tokens=200, batch_size=4):
    """Generate expanded queries using the CoT prompt from Jagerman et al. (2023).
    
    For each query, builds a Chain-of-Thought prompt, runs it through the LLM,
    and concatenates the original query with the model output to form an expanded query.
    Filters out CoT closing phrases as recommended in the paper.

    Args:
        queries (dict): Mapping of query_id (str) to query_text (str).
        model (AutoModelForCausalLM): The loaded HuggingFace causal language model.
        tokenizer (AutoTokenizer): The tokenizer corresponding to the model.
        max_new_tokens (int, optional): Maximum number of new tokens to generate
            per query. Keep below 512. Defaults to 200.
        batch_size (int, optional): Number of queries to process in parallel.
            Reduce if GPU runs out of memory. Defaults to 4.

    Returns:
        dict: Mapping of query_id (str) to expanded_query_text (str),
            where expanded_query_text = original_query + LLM_output.
    """
    expanded_queries = {}
    query_ids = list(queries.keys())
    
    for i in tqdm.tqdm(range(0, len(query_ids), batch_size), desc="Expanding queries"):
        batch_ids = query_ids[i: i + batch_size]
        batch_prompts = [build_cot_prompt(queries[query_id]) for query_id in batch_ids]
        
        # tokenization
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(model.device)
        
        # generate
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=1.0, repetition_penalty=1.1)
            
        # decode new tokens not the prompt here
        for j, query_id in enumerate(batch_ids):
            input_len = inputs["input_ids"].shape[1]
            new_tokens = outputs[j][input_len:]
            expansion = tokenizer.decode(new_tokens, skip_special_tokens=True)
            
            # in paper they have closing phrashes to filter out from CoT output
            expansion = expansion.replace("The final answer:", "").strip()
            expansion = expansion.replace("So the final answer is:", "").strip()
            
            # concatenate orinigal query + LLM output but no 5x repetition
            expanded_queries[query_id] = queries[query_id] + " " + expansion

    return expanded_queries 

In [6]:
# run
expanded_queries = generate_expansions(queries, model, tokenizer)

#check
sample_query_id = list(expanded_queries.keys())[0] # grab fresh one after the filtering
print(f"original:  {queries[sample_query_id]}")
print(f"\nexpanded:  {expanded_queries[sample_query_id]}")

Expanding queries: 100%|██████████| 11/11 [09:16<00:00, 50.61s/it]

original:  do goldfish grow

expanded:  do goldfish grow .

Answer: Yes, goldfish do grow throughout their lives. Goldfish are known to continue growing as long as they live in good conditions with proper nutrition and care. The growth rate of a goldfish may slow down as it reaches maturity, but it will not stop growing entirely. This is because goldfish have an indeterminate growth pattern, meaning they do not reach a fixed size like some other fish species. Instead, they continue to grow based on environmental factors such as food availability, water quality, and genetics. Therefore, it's essential to provide your goldfish with a suitable environment that supports healthy growth while avoiding overfeeding, which can lead to obesity and related health issues.


In [7]:
# loading baseline model for comparison

#rn using pretrained from "sentence-transformers" package
from sentence_transformers import CrossEncoder
CROSS_ENCODER_MODEL ="cross-encoder/ms-marco-MiniLM-L-6-v2" # replace with out own later "path/to/finetune/model"
cross_encoder = CrossEncoder(CROSS_ENCODER_MODEL, max_length=512, device="cuda")
print(f"loaded: {CROSS_ENCODER_MODEL}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

c:\Users\Matiss\anaconda3\envs\py312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Matiss\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back 

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

loaded: cross-encoder/ms-marco-MiniLM-L-6-v2
